In [289]:
import pandas as pd
import glob, os
from tqdm.notebook import tqdm
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import chart_studio
import chart_studio.plotly as py
import plotly.io as pio
import re, string, unicodedata
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', -1)


In [290]:
username = 'politicai' # your username
api_key = 'REDACTED' # your api key - go to profile > settings > regenerate key
chart_studio.tools.set_credentials_file(username=username, api_key=api_key)
# py.plot(fig_object, filename = 'gdp_per_cap', auto_open=True)
# pio.write_html(fig, file=’index.html’, auto_open=True)

In [291]:
def clean_df_format(df):
    df.columns = [x.lower().replace("/", "_").replace(" ", "_").strip() for x in df.columns]
    return df

def clean_country_names(df, column, new_country_col_name="country"):
    df[column] = df[column].str.lower()
    df[column] = df[column].str.replace("(", "").str.replace(")", "").str.replace("&", "")
    df[column] = df[column].str.replace(".", "").str.replace(",", "").str.replace("'","")
    df[column] = df[column].str.replace("-", " ").str.replace(" ", "_").str.replace("__", "_")
    
    name_changes={"korea_north":"north_korea", "korea_dem_people’s_rep":"north_korea", "venezuela_rb":"venezuela",
                  "korea_south":"south_korea","korea_rep":"south_korea" ,"cape_verde":"cabo_verde", "syrian_arab_republic":"syria",
                 "macedonia":"north_macedonia", "serbia_montenegro":"serbia", "slovak_republic":"slovakia",
                 "dem_rep_of_congo":"congo_kinshasa", "congo_dem_rep":"congo_kinshasa","congo_rep":"congo_brazzaville",
                  "egypt_arab_rep":"egypt", "gambia_the":"gambia","the_gambia":"gambia" ,"russian_federation":"russia",
                 "iran_islamic_rep":"iran", "yemen_rep":"yemen", "kyrgyz_republic":"kyrgyzstan", 
                 "lao_pdr":"laos", "bahamas_the":"bahamas", "brunei_darussalam":"brunei", 
                 "hong_kong_sar_china":"hong_kong"}
    df[column] =  df[column].replace(name_changes)

    df = df.rename(columns={column:new_country_col_name})
    return df

# Cargar ficheros

In [292]:
df_fragility = pd.read_excel("../../data/staging/mundial/fragilidad.xls")
df_fragility = clean_df_format(df_fragility)

In [293]:
df_freedomhouse = pd.read_excel("../../data/staging/mundial/freedom_house.xlsx")
df_freedomhouse = clean_df_format(df_freedomhouse)

In [294]:
df_poblacionactiva = pd.read_excel("../../data/staging/mundial/tasa_poblacion_activa.xls")
df_poblacionactiva = clean_df_format(df_poblacionactiva)

In [295]:
df_esperanzavida = pd.read_excel("../../data/staging/mundial/esperanza_vida.xls")
df_esperanzavida = clean_df_format(df_esperanzavida)

In [296]:
df_escolarizacion = pd.read_csv("../../data/staging/mundial/años_escolarizacion_medios.csv", sep=",")
df_escolarizacion = clean_df_format(df_escolarizacion)

In [297]:
df_sistemaselectorales = pd.read_excel("../../data/staging/mundial/sistemas_electorales.xls")
df_sistemaselectorales = clean_df_format(df_sistemaselectorales)

In [298]:
df_poblacion = pd.read_excel("../../data/staging/mundial/poblacion.xls")
df_poblacion = clean_df_format(df_poblacion)

In [299]:
# COUNTRY,YEAR,HOSELECT,HOSTERM,HOSDISS,HOGELECT,HOGTERM,HOGDISS,LEGISL,HOUSENUM,FEDUNIT
#df_constituciones = pd.read_excel("../../data/staging/mundial/constituciones.xlsx")
#df_constituciones = df_constituciones[["country","year","hoselect","hosterm","hosdiss","hogelect","hogterm","hogdiss","legisl","housenum","fedunit"]]
#df_constituciones.to_csv("../../data/staging/mundial/constituciones_reducido.csv", sep=",", index=False)
df_constituciones_red = pd.read_csv("../../data/staging/mundial/constituciones_reducido.csv", sep=",")
df_constituciones_red = clean_df_format(df_constituciones_red)

# Tabla Macro

## Fragility

In [363]:
df_base = df_fragility[["scode", "country", "year", "sfi"]]
df_base = df_base.rename(columns={"scode":"code", "sfi":"state_fragility_idx"})
df_base = clean_country_names(df=df_base, column="country")
print("paises: ", len(df_base.country.unique()))
print("filas: ", df_base.shape)
df_base.head()

paises:  168
filas:  (3960, 4)


,code,country,year,state_fragility_idx
0,AFG,afghanistan,1995,25
1,AFG,afghanistan,1996,24
2,AFG,afghanistan,1997,24
3,AFG,afghanistan,1998,24
4,AFG,afghanistan,1999,24


### Freedom in the world

In [368]:
df_freedomhouse_idx = df_freedomhouse[["country_territory", "status", "total", "edition"]]
df_freedomhouse_idx = df_freedomhouse_idx.rename(columns={"status":"free_not_free", "total":"democracy_idx",
                                                         "edition":"year"})
df_freedomhouse_idx = clean_country_names(df=df_freedomhouse_idx, column="country_territory")
df_freedomhouse_idx.head()

,country,free_not_free,democracy_idx,year
0,abkhazia,PF,40,2020
1,afghanistan,NF,27,2020
2,albania,PF,67,2020
3,algeria,NF,34,2020
4,andorra,F,94,2020


In [369]:
df_base = df_base.merge(right=df_freedomhouse_idx, on=["country", "year"], how="inner")
print("paises: ", len(df_base.country.unique()))
print("filas: ", df_base.shape)
df_base.head()

paises:  165
filas:  (990, 6)


### Poblacion activa

In [334]:
df_poblacionactiva_idx = pd.melt(df_poblacionactiva, id_vars=['country_name','country_code'], value_vars=df_poblacionactiva.columns[4:])
df_poblacionactiva_idx = df_poblacionactiva_idx.rename(columns={"variable":"year"})
df_poblacionactiva_idx = clean_country_names(df=df_poblacionactiva_idx[["country_name", "country_code", "year"]], column="country_name")
df_poblacionactiva_idx.head()

,country,country_code,year
0,aruba,ABW,1960
1,afghanistan,AFG,1960
2,angola,AGO,1960
3,albania,ALB,1960
4,andorra,AND,1960


In [339]:
df_base = df_base.merge(right=df_poblacionactiva_idx, on=["country", "year"], how="inner")
len(df_base.country.unique())

ValueError: You are trying to merge on int64 and object columns. If you wish to proceed you should use pd.concat

In [337]:
no_coinciden = [x for x in df_poblacionactiva_idx.country.unique() if x  in df_base.country.unique()]

In [338]:
len(no_coinciden)

165